In [1]:
from datasets import load_dataset

gsm8k = load_dataset("gsm8k", "main")
print(gsm8k)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [2]:
print(gsm8k["test"].column_names)
print(gsm8k["test"][0])

['question', 'answer']
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [3]:
import pandas as pd
import re

def extract_final_answer(ans):
    # GSM8K format: "... #### 42"
    if "####" in ans:
        return ans.split("####")[-1].strip()
    return ans.strip()

def gsm_to_df(split="test"):
    df = pd.DataFrame(gsm8k[split])
    df["final_answer"] = df["answer"].apply(extract_final_answer)
    return df[["question", "final_answer"]]

gsm_test_df = gsm_to_df("test")
print(gsm_test_df.head())
print("Total test examples:", len(gsm_test_df))

                                            question final_answer
0  Janet’s ducks lay 16 eggs per day. She eats th...           18
1  A robe takes 2 bolts of blue fiber and half th...            3
2  Josh decides to try flipping a house.  He buys...        70000
3  James decides to run 3 sprints 3 times a week....          540
4  Every day, Wendi feeds each of her chickens th...           20
Total test examples: 1319


In [4]:
gsm_test_df.to_csv("gsm8k_test_en_full.csv", index=False)

gsm_test_500 = gsm_test_df.sample(n=500, random_state=42)
gsm_test_500.to_csv("gsm8k_test_en_500.csv", index=False)

print("Saved:", len(gsm_test_df), "full |", len(gsm_test_500), "subset")

Saved: 1319 full | 500 subset


In [5]:
csqa = load_dataset("commonsense_qa")
print(csqa)
print(csqa["train"].column_names)
print(csqa["validation"][0])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})
['id', 'question', 'question_concept', 'choices', 'answerKey']
{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}


In [7]:
print(csqa["validation"][0])
print(type(csqa["validation"][0]["question"]))

{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}
<class 'str'>


In [12]:
def csqa_to_df(split):
    rows = []
    for ex in csqa[split]:
        # question can be either a string or a dict with "stem"
        q = ex["question"]
        stem = q["stem"] if isinstance(q, dict) else q

        choices = None

        if "choices" in ex:
            choices = ex["choices"]
        elif isinstance(q, dict) and "choices" in q:
            choices = q["choices"]
        else:
            raise KeyError("Could not find choices in this dataset example.")

        normalized = []
        if isinstance(choices, list):
            normalized = [(c["label"], c["text"]) for c in choices]
        elif isinstance(choices, dict) and "label" in choices and "text" in choices:
            normalized = list(zip(choices["label"], choices["text"]))
        else:
            raise TypeError(f"Unknown choices format: {type(choices)}")

        options = "\n".join([f"{lab}) {txt}" for lab, txt in normalized])

        rows.append({
            "id": ex.get("id", ex.get("qid", "")),
            "question": stem,
            "options": options,
            "answer_key": ex.get("answerKey", ex.get("answer_key", "")),  # may be missing in test
        })

    return pd.DataFrame(rows)

csqa_val_df = csqa_to_df("validation")
print(csqa_val_df.shape)
csqa_val_df.head(2)

(1221, 4)


,id,question,options,answer_key
0,1afa02df02c908a558b4036e80242fac,A revolving door is convenient for two directi...,A) bank\nB) library\nC) department store\nD) m...,A
1,a7ab086045575bb497933726e4e6ad28,What do people aim to do at work?,A) complete job\nB) learn from each other\nC) ...,A


In [9]:
csqa_val_df.to_csv("commonsenseqa_val_en_full.csv", index=False)

csqa_val_500 = csqa_val_df.sample(n=500, random_state=42)
csqa_val_500.to_csv("commonsenseqa_val_en_500.csv", index=False)

print("Saved CSQA:", len(csqa_val_df), "full |", len(csqa_val_500), "sample")

Saved CSQA: 1221 full | 500 sample


In [10]:
print(csqa["validation"][0].keys())
print(csqa["validation"][0])

dict_keys(['id', 'question', 'question_concept', 'choices', 'answerKey'])
{'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}
